# Inference-time distillation & KL clipping (Snowflake Notebooks · GPU Container Runtime)

OPSD-only demo on **Mistral-7B-Instruct-v0.2** (Apache 2.0, ungated — no HF token).

- **Student**: `mistralai/Mistral-7B-Instruct-v0.2`, trainable, loaded in 4-bit + LoRA so it fits a 24 GB compute pool.
- **Self-teacher**: snapshot of the same Mistral 7B, frozen, served with the **ground-truth answer in its system prompt** (the OPSD privileged-info trick from Section 4 of the essay).

Functions demonstrated:

1. **Section 1** SFT-RS ceiling — record an SFT-RS run as the Section 1 baseline (optional, with Cortex `mistral-large2` as ad-hoc teacher).
2. **Section 4** **OPSD** — student samples without GT; self-teacher computes the per-token reverse KL with GT in context.
3. **Section 5** Gradient geometry — histograms showing OPSD's KL signal is **concentrated** at pivot tokens.
4. **Section 5** **KL clipping** — without it, OPSD collapses inside ~100 steps; with it, stable.
5. **Section 7** Unified meta-algorithm with knobs `(α, λ, π_T)`. The single `unified_step` covers SFT-RS, RL-outcome, OPD, and OPSD by toggling three dials:
    - **α ∈ [0, 1] — RL-outcome weight.** Coefficient on the policy-gradient term `α · R(τ) · ∇ log π_S(τ)`, where `R(τ)` is the scalar verifier reward (1 if the final answer matches gold else 0). `α=0` disables RL entirely → pure imitation. `α=1` turns the rollout into a full RL trajectory rewarded on outcome.
    - **λ ∈ [0, 1] — distillation (KL) weight.** Coefficient on the per-token reverse-KL term `λ · KL(π_T(·|s) ‖ π_S(·|s))` summed over generated positions. `λ=0` removes the teacher entirely → pure RL. `λ=1` weighs the teacher signal at full strength alongside whatever RL is on. The clipped variant `kl.clamp(max=kl_clip)` from Section 5 is what stabilises OPSD.
    - **π_T — teacher identity (categorical).** Which policy plays the teacher in the KL term. `π_T = external_teacher` uses Mixtral-8x7B (Section 4 OPD). `π_T = self_teacher_with_GT` uses the frozen Mistral-7B with the gold answer in its system prompt (Section 5 OPSD). `π_T = student` collapses the KL to zero — used to disable the distillation channel in pure-RL configs.
    
    Concrete instantiations of `(α, λ, π_T)`:
    - **SFT-RS**: `α=0, λ=1, π_T = external_teacher`, sampler = teacher (off-policy on teacher rollouts).
    - **RL-outcome**: `α=1, λ=0, π_T = student` (KL term zero), sampler = student.
    - **OPD**: `α=1, λ=1, π_T = external_teacher`, sampler = student.
    - **OPSD**: `α=1, λ=1, π_T = self_teacher_with_GT`, sampler = student, `kl_clip=1.0`.

**Section 3 OPD with an external same-family teacher is _not_ demonstrated** here — that would need either Mixtral-8x7B (too big for a 24 GB GPU) or a Cortex teacher (no logprobs exposed, so no per-token KL). Pseudocode is shown in Section 7 with the OPD slot disabled.

**Hardware:** GPU compute pool — `GPU_NV_M` (24 GB) is enough with 4-bit + LoRA. Set up via `snowflake_setup.sql`.

**Prereqs:** compute pool `DISTILL_GPU_POOL`, network rule + EAI `HF_EAI` for HuggingFace egress. **No HF token** (Mistral 7B v0.3 is ungated). Optional: External Access Integration is not even strictly required if you point HF cache at a Snowflake stage instead.


## 0 · Setup

In [ ]:
# Container Runtime ships torch/transformers, but the preinstalled bitsandbytes is
# typically too old for transformers' 4-bit code path → ImportError on load_in_4bit=True.
# Force-upgrade bitsandbytes; top up the rest only if missing.
# After this cell runs the FIRST time, restart the kernel so the new bitsandbytes is loaded.
!pip -q install -U "bitsandbytes>=0.43.1"
!pip -q install "transformers>=4.45" "accelerate>=0.30" "peft>=0.11" datasets matplotlib

# ── Hard diagnostic: validate the 4-bit code path BEFORE we try to load a model.
# transformers raises 'Using bitsandbytes 4-bit quantization requires the latest version of
# bitsandbytes' even when bitsandbytes is installed, if any of:
#   (a) torch.cuda.is_available() is False  → wrong compute pool (no GPU)
#   (b) `import bitsandbytes` raises          → torch/CUDA wheel mismatch
#   (c) accelerate not importable             → top-up install above failed
# Surface which one, so the message above isn't misleading.
import importlib, sys
import torch
print('torch                :', torch.__version__)
print('torch CUDA build     :', torch.version.cuda)
print('torch.cuda.available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu                  :', torch.cuda.get_device_name(0))

bnb_import_err = None
try:
    import bitsandbytes as bnb_mod
    print('bitsandbytes         :', bnb_mod.__version__)
except Exception as e:
    bnb_import_err = e
    print('bitsandbytes import FAILED:', repr(e))

try:
    import accelerate as _acc
    print('accelerate           :', _acc.__version__)
except Exception as e:
    print('accelerate import FAILED:', repr(e))

from transformers.utils.import_utils import is_bitsandbytes_available
# CRITICAL: is_bitsandbytes_available is @lru_cache'd. If transformers was imported
# BEFORE the !pip install above (e.g. you ran Section 0 twice in the same kernel), the cached
# False sticks around and you get the 'requires the latest version' ImportError forever.
# Clear the cache so the next call actually re-probes the now-installed bnb wheel.
if hasattr(is_bitsandbytes_available, 'cache_clear'):
    is_bitsandbytes_available.cache_clear()
print('transformers says bnb available:', is_bitsandbytes_available())

# Newer transformers also runs a backend-specific validator. Call it eagerly so the
# preflight catches CUDA-major / bnb-backend mismatches BEFORE from_pretrained does.
try:
    from transformers.integrations.bitsandbytes import validate_bnb_backend_availability
    validate_bnb_backend_availability(raise_exception=True)
    print('bnb backend validator: ok')
except ImportError:
    pass  # older transformers — the main check above is sufficient
except Exception as e:
    raise RuntimeError(
        f'bitsandbytes backend validator rejected the runtime: {e!r}. '
        f'torch={torch.__version__} CUDA={torch.version.cuda}. '
        'Force-reinstall bnb against the runtime CUDA: '
        '`!pip install -U --force-reinstall --no-deps bitsandbytes` then RESTART the kernel.')

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA device visible to torch. 4-bit quantization is GPU-only. '
        'Re-attach this notebook to a GPU compute pool (the metadata hint is '
        'DISTILL_GPU_POOL) and re-run Section 0. If you intentionally want CPU/bf16, set '
        'USE_4BIT=False manually and skip the 4-bit path — but a 7B model in bf16 '
        'will not fit on CPU RAM in this notebook.')
if bnb_import_err is not None:
    raise RuntimeError(
        f'bitsandbytes is installed but `import bitsandbytes` failed: {bnb_import_err!r}. '
        'This usually means the bitsandbytes wheel was built against a different CUDA '
        f'than the runtime torch ({torch.__version__}, CUDA {torch.version.cuda}). Try: '
        '`!pip install -U --force-reinstall --no-deps bitsandbytes` and restart the kernel.')
if not is_bitsandbytes_available():
    raise RuntimeError(
        'transformers reports bitsandbytes unavailable despite the import succeeding above. '
        'Restart the kernel so transformers re-resolves the import cache, then re-run Section 0.')
print('4-bit code path: ready ✓')


In [ ]:
import os, re, math, copy, json, random, time
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
print('device:', DEVICE, '| dtype:', DTYPE)
if DEVICE == 'cuda':
    print('gpu   :', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Mistral-7B-Instruct-v0.2 — Apache 2.0, ungated, 32K vocab (matches Mixtral-8x7B-v0.1).
# Loaded in 4-bit + LoRA so the trainable student fits a 24 GB+ GPU.
import os
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'  # 32K vocab — matches Mixtral-8x7B

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if DEVICE == 'cuda' else 0
USE_4BIT = (vram_gb < 60)
print(f'GPU VRAM ~{vram_gb:.0f} GB → student in {"4-bit + LoRA" if USE_4BIT else "bf16"}')

def load_base(four_bit: bool, prepare_for_training: bool = False):
    if four_bit:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
                                  bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                 device_map='auto')
        if prepare_for_training:
            m = prepare_model_for_kbit_training(
                m,
                use_gradient_checkpointing=True,
                gradient_checkpointing_kwargs={'use_reentrant': False},
            )
        return m
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)

# Student: trainable, prepared for k-bit training (gradient checkpointing wired correctly).
student = load_base(USE_4BIT, prepare_for_training=USE_4BIT)
if USE_4BIT:
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM')
    student = get_peft_model(student, lora)
    # CRITICAL: with 4-bit base + checkpointing, inputs need to require_grad so the
    # autograd graph reaches LoRA params. Otherwise loss.backward() silently zeroes gradients.
    student.enable_input_require_grads()
    student.print_trainable_parameters()

# Self-teacher: separate frozen copy of base model, no LoRA, no training prep.
self_teacher = load_base(USE_4BIT, prepare_for_training=False); self_teacher.eval()
if hasattr(self_teacher, 'gradient_checkpointing_disable'):
    self_teacher.gradient_checkpointing_disable()
for p in self_teacher.parameters(): p.requires_grad_(False)

# Sanity-check: at least one trainable parameter actually has requires_grad=True
trainable = [(n, p) for n, p in student.named_parameters() if p.requires_grad]
assert len(trainable) > 0, 'No trainable params on student — check LoRA config'
print(f'student trainable params: {sum(p.numel() for _,p in trainable)/1e6:.1f} M '
      f'({len(trainable)} tensors)')
print(f'self-teacher params (frozen): {sum(p.numel() for p in self_teacher.parameters())/1e6:.1f} M')
print('same-family precondition: trivially holds (single model, two copies) ✓')


## 1 · Task & verifier (GSM8K subset)

Small grade-school math task. The verifier extracts the final integer after `####` (GSM8K convention) or after `\boxed{...}`.

In [ ]:
ds = load_dataset('gsm8k', 'main', split='train[:48]')
val = load_dataset('gsm8k', 'main', split='test[:512]')

ANS_RE = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?')

def gold_answer(example):
    return example['answer'].split('####')[-1].strip().replace(',', '')

def extract_pred(text: str) -> Optional[str]:
    # last number in the response
    nums = ANS_RE.findall(text.replace(',', ''))
    return nums[-1] if nums else None

def correct(pred: Optional[str], gold: str) -> bool:
    if pred is None: return False
    try: return abs(float(pred) - float(gold)) < 1e-6
    except ValueError: return False

SYS = 'You are a careful math tutor. Reason briefly, then answer with: #### <number>.'

def build_prompt(question: str, answer: Optional[str] = None) -> str:
    msgs = [{'role': 'system', 'content': SYS},
            {'role': 'user',   'content': question}]
    if answer is not None:
        # OPSD privileged-info trick: inject ground truth into teacher context
        msgs[0]['content'] = SYS + f' (For internal grading: the correct final answer is {answer}.)'
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(build_prompt(ds[0]['question'])[:400], '\n...')

## 2 · Helpers — rollout, logprobs, per-token KL

In [ ]:
# All helpers and definitions live here so the slow eval cells below are call-only.
# This cell defines: rollout, logprobs_at, per_token_reverse_kl, wilson_ci, fmt_acc,
# mcnemar_p, bootstrap_gap_ci, difficulty_bucket, evaluate, pass_at_k.
# Cheap to re-run; safe to inspect.

import math
from math import comb
from typing import Sequence, Tuple

MAX_NEW = 160
N_VAL   = 200       # paired-eval n for student / Cortex teachers / local Mixtral

# ---------------------------------------------------------------------------
# Generation + per-token KL  (formerly Section 2 helpers)
# ---------------------------------------------------------------------------
@torch.no_grad()
def rollout(model, prompt: str, temperature: float = 0.7) -> dict:
    inp = tok(prompt, return_tensors='pt').to(DEVICE)
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW,
        pad_token_id=tok.pad_token_id,
        return_dict_in_generate=True,
    )
    if temperature > 0.0:
        gen_kwargs.update(do_sample=True, temperature=temperature)
    else:
        gen_kwargs.update(do_sample=False)  # greedy — do NOT pass temperature
    out = model.generate(**inp, **gen_kwargs)
    full = out.sequences[0]
    plen = inp['input_ids'].shape[1]
    gen  = full[plen:]
    return {'full': full, 'plen': plen, 'gen': gen,
            'text': tok.decode(gen, skip_special_tokens=True)}

def logprobs_at(model, full_ids, plen: int, teacher_prompt_ids=None):
    """Forward-pass; return logprobs at *generated* positions only."""
    if teacher_prompt_ids is None:
        x = full_ids.unsqueeze(0)
        gen_start = plen
    else:
        gen = full_ids[plen:]
        x = torch.cat([teacher_prompt_ids, gen], dim=0).unsqueeze(0)
        gen_start = teacher_prompt_ids.shape[0]
    logits = model(x).logits[0]
    logp   = F.log_softmax(logits.float(), -1)
    return logp[gen_start - 1 : -1]

def per_token_reverse_kl(logp_T, logp_S, clip=None):
    """KL(p_T || p_S) per-position. Optionally point-wise clipped per (t, v)."""
    p_T = logp_T.exp()
    contrib = p_T * (logp_T - logp_S)
    if clip is not None:
        contrib = contrib.clamp(max=clip)
    kl = contrib.sum(-1)
    return kl, kl.detach()

# ---------------------------------------------------------------------------
# Eval statistics  (formerly cell-8b-stats)
# ---------------------------------------------------------------------------
def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """Wilson score 95% CI for a binomial proportion. k=successes, n=trials."""
    if n == 0: return (0.0, 0.0)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half   = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (max(0.0, center - half), min(1.0, center + half))

def fmt_acc(correct_arr: Sequence[int]) -> str:
    n = len(correct_arr); k = int(sum(correct_arr))
    lo, hi = wilson_ci(k, n)
    return f'{k/n:.3f}  [Wilson95: {lo:.3f}, {hi:.3f}]  (n={n})'

def mcnemar_p(a: Sequence[int], b: Sequence[int]) -> dict:
    """Continuity-corrected McNemar paired test. H0: P(a wrong, b right) == P(a right, b wrong)."""
    assert len(a) == len(b), 'McNemar requires paired predictions'
    b01 = sum(1 for x, y in zip(a, b) if x == 0 and y == 1)
    b10 = sum(1 for x, y in zip(a, b) if x == 1 and y == 0)
    if b01 + b10 == 0:
        return {'chi2': 0.0, 'p': 1.0, 'b01': b01, 'b10': b10}
    chi2 = (abs(b01 - b10) - 1) ** 2 / (b01 + b10)
    p = math.erfc(math.sqrt(chi2 / 2))
    return {'chi2': chi2, 'p': p, 'b01': b01, 'b10': b10}

def bootstrap_gap_ci(a: Sequence[int], b: Sequence[int],
                     B: int = 5000, seed: int = 0) -> Tuple[float, float, float]:
    """Bootstrap 95% CI on the paired gap mean(b) - mean(a). Returns (gap, lo, hi)."""
    a_ = np.asarray(a, dtype=int); b_ = np.asarray(b, dtype=int)
    rng = np.random.default_rng(seed)
    n = len(a_)
    idx = rng.integers(0, n, size=(B, n))
    diffs = b_[idx].mean(axis=1) - a_[idx].mean(axis=1)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    return (float(b_.mean() - a_.mean()), float(lo), float(hi))

def difficulty_bucket(ex) -> str:
    """GSM8K difficulty proxy: number of newline-delimited steps in the gold solution."""
    steps = ex['answer'].split('####')[0].count('\n') + 1
    if steps <= 3: return 'easy'
    if steps <= 6: return 'med'
    return 'hard'

# ---------------------------------------------------------------------------
# Eval entrypoints  (moved here so Section 3 / Section 2.5b are call-only)
# ---------------------------------------------------------------------------
def evaluate(model, split, n: int = N_VAL):
    """Greedy eval. Returns (acc, correct_arr) — correct_arr is list[int] for paired tests."""
    was_training = model.training
    model.eval()
    try:
        m = min(n, len(split))
        correct_arr = []
        for ex in split.select(range(m)):
            r = rollout(model, build_prompt(ex['question']), temperature=0.0)
            correct_arr.append(int(correct(extract_pred(r['text']), gold_answer(ex))))
        return sum(correct_arr) / m, correct_arr
    finally:
        if was_training: model.train()

def pass_at_k(arr, k: int):
    """Unbiased pass@k (Chen et al. 2021): per example, 1 - C(n-c, k)/C(n, k).
       arr shape (N, K_total). Returns (mean, per_example_array)."""
    arr = np.asarray(arr)
    n_total = arr.shape[1]
    k = min(k, n_total)
    n_correct = arr.sum(axis=1).astype(int)
    per = []
    for c in n_correct:
        if n_total - c < k:
            per.append(1.0)
        else:
            per.append(1.0 - comb(int(n_total - c), k) / comb(int(n_total), k))
    return float(np.mean(per)), np.asarray(per)

print('helpers ready: rollout, logprobs_at, per_token_reverse_kl, wilson_ci, fmt_acc,')
print('               mcnemar_p, bootstrap_gap_ci, difficulty_bucket, evaluate, pass_at_k')
print(f'               N_VAL={N_VAL}, MAX_NEW={MAX_NEW}')


## 2.4 · Eval methods — quick reference

The notebook uses six eval primitives, all defined in Section 2 helpers and called from Section 3 baseline, Section 2.5 Cortex eval, Section 2.5b local Mixtral, Section 2.6 PRM, and Section 2.7 Pass@k. This cell documents what each one is, why we use it, and when to reach for it.

### 1. Wilson 95% CI — `wilson_ci(k, n)` and `fmt_acc(arr)`

**What.** A 95% confidence interval for a binomial proportion `p = k/n`. Closed-form, two-tailed, slightly asymmetric near 0 and 1.

**Formula** (z = 1.96 for 95%):

$$\hat{p}_{\text{center}} = \frac{p + z^2/2n}{1 + z^2/n} \qquad \text{half-width} = \frac{z\sqrt{p(1-p)/n + z^2/4n^2}}{1 + z^2/n}$$

**Why over the normal approximation** (`p ± z·sqrt(p(1-p)/n)`):
- Normal CI undercovers near `p ∈ {0, 1}` — can produce intervals outside [0, 1].
- Normal CI is too wide near `p = 0.5`.
- Wilson is the recommended interval for binomial proportions in Brown, Cai & DasGupta (2001).

**When.** Every accuracy report. `fmt_acc` wraps it: `0.310 [Wilson95: 0.250, 0.378] (n=200)`.

### 2. McNemar's paired test — `mcnemar_p(a, b)`

**What.** Tests whether two correctness vectors `a, b` (same n examples, same order) disagree symmetrically. Continuity-corrected χ² with df=1:

$$\chi^2 = \frac{(|b_{01} - b_{10}| - 1)^2}{b_{01} + b_{10}}$$

where `b_{01}` = a wrong & b right, `b_{10}` = a right & b wrong. Diagonal cells (both right / both wrong) are ignored — that's the point.

**Why over an unpaired two-proportion test.** When the same examples are graded by two systems, paired tests have lower variance because shared difficulty across examples drops out. Two unpaired Wilson CIs that *overlap* can still correspond to a McNemar p < 0.001 — paired tests detect signals that marginal CIs hide.

**When.** Comparing the student to each Cortex teacher (Section 2.5), and the student to the local Mixtral teacher (Section 2.5b). Only valid when both vectors come from the same `val.select(range(N))` slice in the same order.

**Returned fields.** `chi2`, `p` (two-sided), `b01`, `b10`. Significance stars at the call site: `*** < 0.001 < ** < 0.01 < * < 0.05 < ns`.

### 3. Bootstrap CI on the gap — `bootstrap_gap_ci(a, b, B=5000)`

**What.** Resamples `(a, b)` with replacement `B = 5000` times, recomputes the paired gap `mean(b) - mean(a)` per resample, returns the 2.5th/97.5th percentiles. `(gap, lo, hi)`.

**Why both bootstrap CI and McNemar.** They answer different questions:
- McNemar: *is the gap significantly different from zero?* → p-value
- Bootstrap CI: *what's the plausible range of the gap?* → effect-size band

A McNemar p < 0.05 with a bootstrap CI of `[+0.005, +0.060]` says "yes, significantly above zero, but tiny". You need both.

**When.** Any time you report a gap. Used in Section 2.5 (teacher-vs-student, teacher-vs-teacher) and Section 2.5b (Mixtral-vs-student).

### 4. Pass@k — `pass_at_k(arr, k)`

**What.** Unbiased estimator from Chen et al. 2021 (HumanEval): probability that a sample of size `k` drawn without replacement from `K` student rollouts contains at least one correct. Per example with `c` corrects out of `K`:

$$\text{pass@}k(\text{example}) = 1 - \binom{K - c}{k} \Big/ \binom{K}{k}$$

Mean across examples is the reported `pass@k`.

**Why.** Three uses:
- **Best-of-K SFT ceiling.** Pass@k is the upper bound for any rejection-sampling SFT loop using `k`-way student-self rollouts.
- **Saturation diagnostic.** If `pass@K ≈ pass@1`, the student is saturated at the chosen temperature; distillation is the only path forward. If `pass@K ≫ pass@1`, RS-SFT can extract the gap without an external teacher.
- **Headroom estimate.** `pass@K - pass@1` ≈ how much accuracy you can recover by filtering correct rollouts.

**When.** Section 2.7. Default `K=8`, `N_PASSK=64`, `temp=0.7`. Bootstrap CI over examples (B=2000) reported alongside each `pass@k` value.

**Watch out.** Pass@k mixes the temperature you sample at. Reporting `pass@1` at `temp=0.7` is *not* the greedy baseline — the greedy baseline is `temp=0.0` (= Section 3 `acc0`). The two are related but different; the gap between them is a temperature-tuning signal.

### 5. PRM (Process Reward Model) judge — `prm_judge(rollout_text, ex, agg)`

**What.** LLM-as-judge that grades a reasoning chain step-by-step and aggregates per-step scores into a scalar reward in `[0, 1]`. Currently a **stub**, not paper-faithful — the rubric, segmentation, and aggregation are placeholders.

**Components.**
- `PRM_JUDGE_MODEL`: the Cortex model that does the grading. Default `'llama4-maverick'` because it scored highest in Section 2.5.
- `PRM_RUBRIC`: system prompt asking for per-step `{step, score, reason}` JSON, then a final `{steps, min, mean}` summary.
- `split_steps_naive`: local fallback that splits on numbered steps `\d+[.)]` or blank lines. Only invoked if the judge response can't be parsed.
- `agg`: aggregation rule.
  - `'min'` → product-of-correctness. Any wrong step ⇒ 0. Strict, paper-typical for math PRMs.
  - `'mean'` → fraction of correct steps. Smoother gradient if used as RL reward.
  - `'last'` → last-step score, ≈ final-answer correctness.

**Two ways to use it (both currently stubbed in Section 2.6).**
1. **RS-SFT filter**: keep rollouts where `prm_judge(text, ex) > τ`, SFT student on those.
2. **Reward replacement**: drop `reward_fn = lambda t, ex: int(correct(...))` and use `reward_fn = prm_judge` in Section 9 `unified_step`. RL/OPD now optimise for *process correctness*, not just final-answer match.

**When.** Domain-specific. Math reasoning (GSM8K, MATH) is where PRMs shine because step-level errors are detectable. For tasks with no decomposable reasoning chain, PRMs degenerate to outcome rewards.

**Currently incomplete.** Needs the paper's exact rubric, segmentation rule, and aggregation. The stub is meant as a plug point; replace `PRM_RUBRIC` and the `split_steps_naive` / `agg` defaults with the paper's spec.

### 6. Difficulty bucketing — `difficulty_bucket(ex)`

**What.** GSM8K-specific proxy: counts `\n`-delimited lines in the gold solution (excluding `####`). Bins into `easy ≤ 3 steps`, `med ≤ 6`, `hard > 6`.

**Why.** A teacher's overall accuracy gain might be entirely concentrated in one bucket (e.g. `llama4-maverick` is +0.30 on hard but only +0.05 on easy). Reporting bucketed accuracy + Wilson CIs surfaces this; reporting only the marginal gap hides it.

**When.** Section 2.5 prints a per-bucket Wilson-CI table after the main eval.

---

### Decision guide — which method to reach for

| Question | Method | Where used |
|---|---|---|
| What's the accuracy and how confident am I? | Wilson 95% CI via `fmt_acc` | Section 3, Section 2.5, Section 2.5b |
| Is teacher A really better than student B? | McNemar paired test | Section 2.5, Section 2.5b |
| How big is the gap, with a CI? | Paired bootstrap | Section 2.5, Section 2.5b |
| Where in the difficulty distribution is the gap? | `difficulty_bucket` + Wilson per bucket | Section 2.5 |
| What's the SFT-RS ceiling without an external teacher? | Pass@k | Section 2.7 |
| Should I reward final answers or reasoning steps? | PRM judge (when implemented) | Section 2.6 (stub) |
| Is the gap practically significant or just statistically? | Bootstrap CI half-width vs. effect-size threshold | manual interpretation |

### What we deliberately don't do (and why)

- **No t-tests on accuracy.** Accuracy is binomial, not Gaussian; t-tests on Bernoulli proportions are inferior to Wilson + McNemar.
- **No multiple-comparisons correction.** With 2-3 teachers and one test family (paired-vs-student), Bonferroni would be overcautious. Worth adding once the teacher count grows.
- **No calibration metrics (ECE, Brier).** Accuracy doesn't expose them; we'd need token-level confidence from `rollout`, which is local-models-only (Cortex teachers don't expose logprobs).
- **No sample-efficiency curves.** All evals run at fixed n; the budget gain from `n=24 → n=200` is the only sample-size move we make explicitly.


## 3 · Baseline — student before any training
Quick smoke test to confirm the pipeline works and to record a starting accuracy.

In [ ]:
# Section 3 baseline — call-only. All defs are in Section 2 helpers.
# Greedy eval on N_VAL=200 GSM8K val problems. Wall ~10-15 min on the GPU pool.

# Fail FAST: dependency check before the slow loop.
for _required in ('fmt_acc', 'wilson_ci', 'evaluate'):
    if _required not in dir():
        raise NameError(f'{_required} is undefined. Run Section 2 helpers first.')

acc0, student_correct = evaluate(student, val, n=N_VAL)
print(f'student baseline acc: {fmt_acc(student_correct)}')


## 2.5 · Cortex REST API — reference teacher accuracy ceiling (Section 1)

Section 1 of the essay frames SFT/RS as bounded by the teacher\'s accuracy. We make that ceiling concrete: hit Cortex on the same GSM8K val subset with strong hosted models and record their accuracy.

### How the calls go out

- **Auth**: through the active Snowpark session — `get_active_session().connection.rest.token` provides a live Snowflake session token that the connector refreshes transparently. Sent as `Authorization: Snowflake Token="<token>"`. No PATs in code.
- **Routing**: **direct HTTP** to `POST /api/v2/cortex/inference:complete` on the account host. **Not** a SQL call to `SNOWFLAKE.CORTEX.COMPLETE()` — the previous version used `session.sql(...)`, which incurs SQL parse + warehouse + result-set overhead per request. The new client streams Server-Sent Events back and aggregates them client-side.
- **Limitation**: Cortex doesn\'t expose `logprobs`, so this teacher can\'t drive OPD\'s per-token reverse KL — only a **text-output reference** (and an SFT-RS data source if you want to extend).

### REST API call inventory across the notebook

All API traffic flows through one entrypoint: `client.complete(messages, model, max_tokens, temperature)` on the `CortexRESTClient` defined in the cell below. The wire-level request looks like:

```
POST https://<account>.snowflakecomputing.com/api/v2/cortex/inference:complete
Authorization: Snowflake Token="<session-token>"
Content-Type: application/json
Accept: text/event-stream

{
  "model": "<model>",
  "messages": [{"role": "system", "content": "..."},
               {"role": "user",   "content": "..."}],
  "max_tokens": 160,
  "temperature": 0.0
}
```

The response is a stream of SSE events:

```
data: {"choices": [{"delta": {"content": "Hel"}}], "model": "..."}
data: {"choices": [{"delta": {"content": "lo"}}]}
...
data: {"usage": {"prompt_tokens": 27, "completion_tokens": 14}}
data: [DONE]
```

`_parse_sse` aggregates `choices[0].delta.content` chunks into a single string and surfaces the final `usage` block.

| Where | Purpose | Model | Per-call options | Volume per run |
|---|---|---|---|---|
| Section 2.5 client init (cell-12) | Sanity ping at client construction | `claude-sonnet-4-6` (default) | `max_tokens=8`, `temperature=0.0` | 1 call |
| Section 2.5 Cortex eval (cell-13) | Teacher accuracy on GSM8K val | each of `CORTEX_MODELS` (e.g. `mistral-large2`, `llama4-maverick`) | `max_tokens=MAX_NEW=160`, `temperature=0.0` (greedy) | `N_VAL_CORTEX × len(CORTEX_MODELS)` calls — at the defaults: **200 × 2 = 400 calls** |
| Section 2.6 PRM judge (cell-13c-prm-code) | Step-by-step reasoning grading via LLM-as-judge | `PRM_JUDGE_MODEL = \'llama4-maverick\'` | `max_tokens=512`, `temperature=0.0` | 1 call per `prm_judge(rollout_text, ex)` invocation. Smoke test: 1 call. If wired into RS-SFT filter for the full val set: up to `N_VAL` calls. |

### Cost / token accounting

Token usage is harvested from the `usage` SSE event and tracked in `cortex_usage[model] = (total_in, total_out)` in the Section 2.5 eval cell, then reported as `out-tok/correct = total_out / max(correct_n, 1)` so teachers are ranked by accuracy-per-token, not just accuracy.

### Failure modes the client handles

- **Non-200 HTTP** → `{\'error\': \'HTTP 4xx: <body>\', \'elapsed\': ...}` instead of raising. Eval loops fall through to `extract_pred(\'\')` which returns `None`, scoring as wrong. A transient Cortex error costs one wrong-answer unit, not the whole loop.
- **SSE stream interruption / network error** → `{\'error\': str(e), \'elapsed\': ...}`. Same fallthrough as HTTP errors.
- **Malformed JSON in judge response** (PRM only) → `parse_llm_json()` retries plain JSON, fenced JSON, then a brace-greedy regex; only after all three fail does it return `{\'_parse_error\': True, ...}` and `prm_judge` falls through to `0.0`.

### Additional notes

- Section 2 helpers `rollout`, `evaluate`, `pass_at_k` — local GPU inference on the LoRA student.
- Section 4 Mixtral-8x7B teacher — local GPU 4-bit, not Cortex.
- Section 5 self-teacher — local GPU, frozen copy of student.
- Section 7 KL-clipping training loop — local backprop, no API traffic.

So the only network egress in the entire notebook is the Cortex client. Easy to firewall, easy to budget.

The REST path is the canonical Cortex inference interface and what production agents use. 


In [ ]:
# Cortex REST API client — DIRECT HTTP, no SQL roundtrip.
#
# Endpoint: POST https://<account>.snowflakecomputing.com/api/v2/cortex/inference:complete
# Auth:     Snowflake session token from the active Snowpark connection
#           (Authorization: Snowflake Token="<token>")
# Body:     OpenAI-style {model, messages, max_tokens, temperature}
# Response: Server-Sent Events (text/event-stream). Aggregated client-side into a
#           single {content, usage, model} dict so the caller interface is unchanged.
import json, time, re, requests
from typing import List, Dict, Optional
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f'Connected: {session.get_current_account()}')

class CortexRESTClient:
    """Direct REST client for Snowflake Cortex inference.

    Why HTTP and not session.sql(SNOWFLAKE.CORTEX.COMPLETE(...)):
      * No SQL parsing / no warehouse spin / no result-set materialization
      * Native streaming (SSE) so first-token latency is measurable
      * Standard headers + JSON body — easy to firewall, easy to budget
      * Identical OpenAI-shaped messages array; no JSON-in-SQL escaping
    """

    INFERENCE_PATH = '/api/v2/cortex/inference:complete'

    def __init__(self, default_model: str = 'claude-sonnet-4-6', timeout: int = 120):
        self.default_model = default_model
        self.timeout = timeout
        # Resolve the account host from the live Snowpark connection.
        conn = session.connection
        host = getattr(conn, 'host', None) or session.get_current_account()
        if not host.startswith('http'):
            host = 'https://' + host
        self.base_url = host.rstrip('/')
        self.url      = self.base_url + self.INFERENCE_PATH

    def _headers(self) -> Dict[str, str]:
        # Live session token; the connector refreshes it transparently.
        token = session.connection.rest.token
        return {
            'Authorization': f'Snowflake Token="{token}"',
            'Content-Type':  'application/json',
            'Accept':        'text/event-stream',
        }

    @staticmethod
    def _parse_sse(resp) -> Dict:
        """Aggregate SSE deltas into one completion. Returns {content, usage, model}."""
        parts: List[str] = []
        usage: Dict = {}
        model_name: Optional[str] = None
        for line in resp.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data:'):
                continue
            payload = line[len('data:'):].strip()
            if payload == '[DONE]':
                break
            try:
                ev = json.loads(payload)
            except json.JSONDecodeError:
                continue
            if 'model' in ev and not model_name:
                model_name = ev['model']
            if 'usage' in ev and ev['usage']:
                usage = ev['usage']
            choice = (ev.get('choices') or [{}])[0]
            delta  = choice.get('delta') or {}
            chunk  = delta.get('content')
            if chunk:
                parts.append(chunk)
            # Some servers emit non-streamed shape inside an event
            msg = choice.get('message') or {}
            if not chunk and msg.get('content'):
                parts.append(msg['content'])
        return {'content': ''.join(parts), 'usage': usage, 'model': model_name}

    def complete(self, messages: List[Dict], model: str = None,
                 max_tokens: int = 2048, temperature: float = 0.0) -> Dict:
        body = {
            'model':       model or self.default_model,
            'messages':    messages,
            'max_tokens':  max_tokens,
            'temperature': temperature,
        }
        t0 = time.time()
        try:
            with requests.post(self.url, headers=self._headers(),
                               json=body, stream=True, timeout=self.timeout) as r:
                elapsed = time.time() - t0
                if r.status_code != 200:
                    return {'error': f'HTTP {r.status_code}: {r.text[:500]}',
                            'elapsed': elapsed}
                parsed = self._parse_sse(r)
                return {'content': parsed['content'],
                        'usage':   parsed['usage'],
                        'model':   parsed['model'] or body['model'],
                        'elapsed': time.time() - t0}
        except Exception as e:
            return {'error': str(e), 'elapsed': time.time() - t0}

    def complete_text(self, system: str, user: str, **kwargs) -> str:
        msgs = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': user}]
        r = self.complete(msgs, **kwargs)
        return r.get('content') or r.get('error') or 'No response'


def parse_llm_json(text: str) -> dict:
    """Parse JSON from an LLM response, handling fences and preamble."""
    if not text or not text.strip():
        return {}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fenced = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', text, re.DOTALL)
    if fenced:
        try:
            return json.loads(fenced.group(1).strip())
        except json.JSONDecodeError:
            pass
    brace = re.search(r'(\{.*\})', text, re.DOTALL)
    if brace:
        try:
            return json.loads(brace.group(1))
        except json.JSONDecodeError:
            pass
    return {'_parse_error': True, '_raw': text[:500]}


client = CortexRESTClient()

# Sanity ping — exercises the full HTTP + SSE path.
_t = client.complete_text('You are a helpful assistant.',
                          'Say hello in one word.', max_tokens=8)
print(f'Cortex REST API response: {_t}')


In [ ]:
# Evaluate Cortex teachers on the same GSM8K val subset used for the student baseline.
# Same metric, same n, same SYS prompt — DIRECTLY paired with student_correct so we can run
# McNemar (paired) and bootstrap CI on the gap.
import time

CORTEX_MODELS = ['mistral-large2', 'llama4-maverick']  # add 'llama3.1-70b' / 'claude-sonnet-4-5' if you want
N_VAL_CORTEX  = N_VAL  # MUST match the baseline n for paired stats

# Fail FAST before the 25-min-per-model loop: confirm the stats utilities are loaded.
# This was the gotcha that cost 47 min last time — print uses fmt_acc, undefined ⇒ NameError
# AFTER the API calls already ran. We check up front instead.
for _required in ('fmt_acc', 'mcnemar_p', 'bootstrap_gap_ci', 'wilson_ci', 'difficulty_bucket'):
    if _required not in dir():
        raise NameError(
            f'{_required} is undefined. Run the eval-stats-utilities cell '
            '(between Section 2 helpers and Section 3) BEFORE this cell. It defines fmt_acc, mcnemar_p, '
            'bootstrap_gap_ci, wilson_ci, difficulty_bucket.')
if 'student_correct' not in dir():
    raise NameError('student_correct is undefined. Run the Section 3 baseline cell first.')

# Idempotent: persist results across runs so a crash mid-loop doesn\'t cost re-querying.
if 'cortex_correct' not in dir(): cortex_correct = {}
if 'cortex_pred'    not in dir(): cortex_pred    = {}
if 'cortex_acc'     not in dir(): cortex_acc     = {}
if 'cortex_usage'   not in dir(): cortex_usage   = {}

for m in CORTEX_MODELS:
    if m in cortex_correct and len(cortex_correct[m]) == N_VAL_CORTEX:
        # Already evaluated at the right n — skip the API calls, just reprint.
        ti, to = cortex_usage.get(m, (0, 0))
        print(f'{m:18s} (cached)  acc: {fmt_acc(cortex_correct[m])}   '
              f'tokens in/out = {ti}/{to}')
        continue
    arr, preds = [], []
    total_in = total_out = 0
    t0 = time.time()
    for ex in val.select(range(N_VAL_CORTEX)):
        msgs = [{'role':'system','content': SYS},
                {'role':'user',  'content': ex['question']}]
        r = client.complete(msgs, model=m, max_tokens=MAX_NEW, temperature=0.0)
        text = r.get('content', '') or r.get('error', '')
        u    = r.get('usage', {}) or {}
        total_in  += u.get('prompt_tokens', 0)
        total_out += u.get('completion_tokens', 0)
        preds.append(text)
        arr.append(int(correct(extract_pred(text), gold_answer(ex))))
    cortex_correct[m] = arr
    cortex_pred[m]    = preds
    cortex_acc[m]     = sum(arr) / N_VAL_CORTEX
    cortex_usage[m]   = (total_in, total_out)
    cost_per_correct  = total_out / max(sum(arr), 1)
    print(f'{m:18s} acc: {fmt_acc(arr)}   '
          f'tokens in/out = {total_in}/{total_out}   '
          f'out-tok/correct = {cost_per_correct:.0f}   '
          f'wall = {time.time()-t0:.1f}s')

print()
print(f'student baseline   {fmt_acc(student_correct)}')
print()

# Paired teacher-vs-student tests — the only comparison that matters at this n.
for m, arr in cortex_correct.items():
    mn = mcnemar_p(student_correct, arr)
    gap, glo, ghi = bootstrap_gap_ci(student_correct, arr)
    sig = '***' if mn['p'] < 0.001 else ('**' if mn['p'] < 0.01 else ('*' if mn['p'] < 0.05 else 'ns'))
    print(f'{m:18s} vs student: gap={gap:+.3f} [boot95: {glo:+.3f}, {ghi:+.3f}]   '
          f'McNemar χ²={mn["chi2"]:.2f} p={mn["p"]:.4f} {sig}   '
          f'(student-only-right={mn["b10"]}, teacher-only-right={mn["b01"]})')

# Pairwise teacher-vs-teacher
print()
ms = list(cortex_correct.keys())
for i in range(len(ms)):
    for j in range(i+1, len(ms)):
        a, b = cortex_correct[ms[i]], cortex_correct[ms[j]]
        mn = mcnemar_p(a, b); gap, glo, ghi = bootstrap_gap_ci(a, b)
        print(f'{ms[j]:18s} vs {ms[i]:18s}: gap={gap:+.3f} [{glo:+.3f},{ghi:+.3f}]  McNemar p={mn["p"]:.4f}')

# Per-difficulty bucketing — gap might be concentrated in one tier
buckets = [difficulty_bucket(val[i]) for i in range(N_VAL_CORTEX)]
print()
print('Per-difficulty accuracy (Wilson95 CI):')
tiers = ['easy', 'med', 'hard']
header = f"{'model':22s} " + ' '.join(f'{b:>22s}' for b in tiers)
print(header)
all_models = [('student', student_correct)] + list(cortex_correct.items())
for name, arr in all_models:
    row = []
    for tier in tiers:
        sub = [c for c, b in zip(arr, buckets) if b == tier]
        if not sub:
            row.append('—')
        else:
            lo, hi = wilson_ci(sum(sub), len(sub))
            row.append(f'{sum(sub)/len(sub):.2f} [{lo:.2f},{hi:.2f}] n={len(sub)}')
    print(f'{name:22s} ' + ' '.join(f'{r:>22s}' for r in row))

print()
print('Reading the table:')
print('  • Wilson CIs replace point estimates — overlapping CIs ⇒ noise, not signal.')
print('  • McNemar tests the *paired* H0; small p ⇒ teacher and student really disagree')
print('    on different examples, not just at marginally different rates.')
print('  • out-tok/correct ranks teachers by acc/cost — strict accuracy can be misleading.')
print('  • Cortex teachers cannot drive OPD/OPSD (no logprobs). Use them as RS-SFT data')
print('    sources: keep rollouts where the teacher solved correctly, SFT student on those.')


### 2.6 · PRM (process reward model) judge

A process reward model scores the *reasoning chain* step-by-step, then aggregates the per-step scores into a scalar reward in `[0, 1]`. Two design choices distinguish a useful PRM from a stub:

1. **Deterministic step segmentation.** Steps are split in Python (`split_steps`) before being sent to the judge — not chosen by the judge itself. This stabilises `min`/`mean` aggregation across runs and prevents the judge from collapsing the whole chain into one "step".
2. **Hash-keyed cache.** `_prm_cache` memoises results by `(judge_model, rubric, question, steps)` hash, so re-running with a different aggregation rule (`min` ↔ `mean` ↔ `last`) costs zero extra Cortex calls.

### Calibrate before you trust it

A PRM judge is just an LLM-as-judge — it can be miscalibrated, biased toward particular phrasings, or insensitive to the errors that actually matter on your task. Before using `prm_judge` in any training loop, run **Section 2.6b** below: it computes Spearman ρ and AUROC of `prm_judge(min/mean/last)` against verifier ground truth on the existing `cortex_pred` rollouts. Targets:

- **AUROC > 0.85**: judge separates correct from incorrect rollouts well; safe to use.
- **AUROC 0.7–0.85**: judge has signal but is noisy; ensemble it (Step E in plan).
- **AUROC < 0.7**: rubric isn\'t fit for the task; rewrite the rubric or pick a different judge model.

### Plug points (use after calibration passes)

1. **RS-SFT filter** — keep only rollouts where `prm_judge(text, ex, agg=\'mean\') ≥ τ`, then SFT student on those.
2. **`reward_fn` replacement** — drop `reward_fn = lambda t, ex: int(correct(...))` for `reward_fn = lambda t, ex: prm_judge(t, ex, agg=\'mean\')` in the Section 9 unified meta-algorithm. RL-outcome and OPD then optimise for *process correctness*, not just final-answer match.

### Knobs you can tune

- `PRM_JUDGE_MODEL`: which Cortex model grades. Default `llama4-maverick`. Switch to a stronger or cheaper model based on calibration AUROC.
- `PRM_RUBRIC`: the per-step scoring criterion. Three variants worth ablating against AUROC: strict-arithmetic, logical-progression, combined (current default).
- `agg`: `min` for strict gating, `mean` for smooth RL reward, `last` for outcome-only behavior.


In [ ]:
# PRM judge — non-stub. Deterministic pre-segmentation + hash-keyed cache.
#
# Pipeline:
#   1. split_steps(rollout) → list[str]   (Python-side, deterministic)
#   2. judge sees a fixed (step_id, step_text) list and returns one 0/1 per step
#   3. _prm_cache memoises by (model, rubric, question, steps) hash
#   4. prm_judge(text, ex, agg) aggregates the per-step scores
#
# Calibrate against ground-truth correctness in the next cell before relying on it.

import hashlib

PRM_JUDGE_MODEL = 'llama4-maverick'  # strongest non-Mistral teacher in the Section 2.5 eval

PRM_RUBRIC = (
    'You are grading a math reasoning chain. The reasoning has been pre-split into numbered steps.\n'
    'For EACH step in the user message, output a JSON object: {"step": <int>, "score": 0 or 1}.\n'
    'Score 1 iff (a) any arithmetic in the step is exact AND (b) the step follows logically from prior steps.\n'
    'Score 0 if there is any arithmetic error, dropped variable, or unjustified leap.\n'
    'Return a single JSON array of step objects, in step order. No other text, no prose.\n'
    'Example: [{"step": 1, "score": 1}, {"step": 2, "score": 0}, {"step": 3, "score": 1}]'
)

def split_steps(rollout_text: str) -> list:
    """Deterministic step segmentation for GSM8K-style chain-of-thought.
    
    Cascade:
      1. Numbered-step regex (1., 2), Step 3:, ...).
      2. Blank-line split (paragraph breaks).
      3. Sentence split on .!? followed by capital letter.
    
    Strips the final '#### N' answer line — that's graded by the verifier, not the PRM.
    """
    text = rollout_text.strip()
    text = re.sub(r'\n*####\s*-?[\d,.]+\s*$', '', text).strip()
    if not text:
        return []
    
    parts = re.split(r'\n(?=(?:Step\s*)?\d+[\.\):]\s)', text)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) >= 2:
        return parts
    
    parts = re.split(r'\n\s*\n', text)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) >= 2:
        return parts
    
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [text]

_prm_cache = {}  # hash → list[int]

def _cache_key(model: str, rubric: str, question: str, steps: list) -> str:
    blob = json.dumps({'m': model, 'r': rubric, 'q': question, 's': steps}, sort_keys=True)
    return hashlib.md5(blob.encode()).hexdigest()

def _prm_score_steps(rollout_text: str, ex: dict) -> list:
    """Returns list[int] of 0/1 scores, one per pre-segmented step. Cached."""
    steps = split_steps(rollout_text)
    if not steps:
        return []
    key = _cache_key(PRM_JUDGE_MODEL, PRM_RUBRIC, ex['question'], steps)
    if key in _prm_cache:
        return _prm_cache[key]
    
    formatted = '\n'.join(f'Step {i+1}: {s}' for i, s in enumerate(steps))
    msgs = [
        {'role': 'system', 'content': PRM_RUBRIC},
        {'role': 'user',   'content': f'Problem:\n{ex["question"]}\n\nSteps to grade:\n{formatted}'},
    ]
    r = client.complete(msgs, model=PRM_JUDGE_MODEL, max_tokens=512, temperature=0.0)
    parsed = parse_llm_json(r.get('content', '') or '')
    
    if isinstance(parsed, list):
        items = parsed
    elif isinstance(parsed, dict):
        items = parsed.get('steps') or []
    else:
        items = []
    
    scores = []
    for item in items:
        if isinstance(item, dict):
            scores.append(int(item.get('score', 0)))
    
    # Defensive pad/truncate to match step count (judge sometimes drops the last step)
    if len(scores) < len(steps):
        scores += [0] * (len(steps) - len(scores))
    elif len(scores) > len(steps):
        scores = scores[:len(steps)]
    
    _prm_cache[key] = scores
    return scores

def prm_judge(rollout_text: str, ex: dict, agg: str = 'min') -> float:
    """Returns scalar reward in [0, 1].
       agg='min'  → product of step scores. Strict; any wrong step ⇒ 0.
       agg='mean' → fraction of correct steps. Smooth; usable as RL reward.
       agg='last' → last-step score, ≈ final-answer correctness.
    """
    scores = _prm_score_steps(rollout_text, ex)
    if not scores:
        return 0.0
    if agg == 'min':  return float(min(scores))
    if agg == 'mean': return sum(scores) / len(scores)
    if agg == 'last': return float(scores[-1])
    raise ValueError(f'unknown agg={agg!r}')

# Smoke test on one student rollout — exercises split + judge + cache.
_ex = val[0]
_r  = rollout(student, build_prompt(_ex['question']), temperature=0.0)
_steps = split_steps(_r['text'])
print('PRM judge smoke test:')
print(f'  question  : {_ex["question"][:80]}...')
print(f'  rollout   : {_r["text"][:120]}...')
print(f'  segmented into {len(_steps)} step(s)')
print(f'  prm(min)  = {prm_judge(_r["text"], _ex, agg="min"):.2f}')
print(f'  prm(mean) = {prm_judge(_r["text"], _ex, agg="mean"):.2f}  (cache hit on 2nd call ⇒ free)')
print(f'  prm(last) = {prm_judge(_r["text"], _ex, agg="last"):.2f}')
print(f'  cache size: {len(_prm_cache)} entries')


### 2.6b · PRM calibration — Spearman ρ + AUROC vs verifier

Runs `prm_judge` on the existing `cortex_pred[m]` rollouts (collected in Section 2.5) and reports correlation/separation against `cortex_correct[m]` (the verifier ground truth). This is what proves the judge has signal — without it, `prm_judge` is just an expensive random number generator.

For each Cortex model and each aggregation rule (`min`, `mean`, `last`):

- **Spearman ρ**: rank correlation between PRM score and verifier correctness. ρ > 0.5 is decent, > 0.7 is good.
- **AUROC**: probability that PRM ranks a correct rollout above an incorrect one. > 0.85 is the bar for "safe to use as a reward".

If both metrics are weak across all aggregations, fix the rubric before wiring `prm_judge` into RS-SFT or `unified_step`.


In [ ]:
# Section 2.6b — PRM calibration. ~5 sec per rollout × 200 × 2 models ≈ 30 min on first run;
# subsequent runs are free thanks to _prm_cache.

# Fail FAST on missing prereqs.
for _r in ('prm_judge', 'cortex_pred', 'cortex_correct', 'val'):
    if _r not in dir():
        raise NameError(f'{_r} undefined. Run Section 2.5 (Cortex eval) and Section 2.6 (PRM defs) first.')
if not cortex_pred:
    raise RuntimeError('cortex_pred is empty. Run the Section 2.5 Cortex eval cell first.')


def spearman_rho(x, y) -> float:
    """Spearman rank correlation with average-rank ties."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 2: return 0.0
    def avg_rank(a):
        order = np.argsort(a, kind='stable')
        ranks = np.empty(n, dtype=float)
        i = 0
        while i < n:
            j = i
            while j + 1 < n and a[order[j+1]] == a[order[i]]: j += 1
            r = (i + j) / 2.0 + 1.0
            for k in range(i, j+1): ranks[order[k]] = r
            i = j + 1
        return ranks
    rx, ry = avg_rank(x), avg_rank(y)
    sx, sy = rx.std(), ry.std()
    if sx == 0 or sy == 0: return 0.0
    return float(((rx - rx.mean()) * (ry - ry.mean())).mean() / (sx * sy))


def auroc(scores, labels) -> float:
    """AUROC via Mann-Whitney U with average-rank tie handling."""
    s = np.asarray(scores, dtype=float); l = np.asarray(labels, dtype=int)
    pos_mask = l == 1
    n_pos, n_neg = int(pos_mask.sum()), int((~pos_mask).sum())
    if n_pos == 0 or n_neg == 0: return 0.5
    order = np.argsort(s, kind='stable')
    ranks = np.empty_like(s)
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[order[j+1]] == s[order[i]]: j += 1
        r = (i + j) / 2.0 + 1.0
        for k in range(i, j+1): ranks[order[k]] = r
        i = j + 1
    rank_sum_pos = ranks[pos_mask].sum()
    U = rank_sum_pos - n_pos * (n_pos + 1) / 2.0
    return float(U / (n_pos * n_neg))


import time
print(f'Calibrating PRM judge ({PRM_JUDGE_MODEL}) against verifier on {sum(len(v) for v in cortex_pred.values())} rollouts...')
print()

prm_scores = {}  # {model: {agg: list[float]}}
for m, preds in cortex_pred.items():
    print(f'  scoring {m}: {len(preds)} rollouts ', end='', flush=True)
    t0 = time.time()
    by_agg = {'min': [], 'mean': [], 'last': []}
    for k, text in enumerate(preds):
        ex = val[k]
        for agg in by_agg:
            by_agg[agg].append(prm_judge(text, ex, agg=agg))
        if (k+1) % 25 == 0: print('.', end='', flush=True)
    prm_scores[m] = by_agg
    print(f' done in {time.time()-t0:.1f}s   cache size: {len(_prm_cache)}')

print()
print(f'{"model":18s} {"agg":>6s}  {"Spearman ρ":>11s}  {"AUROC":>7s}  verdict')
print('-' * 72)
for m, scores_by_agg in prm_scores.items():
    labels = cortex_correct[m]
    for agg in ('min', 'mean', 'last'):
        rho = spearman_rho(scores_by_agg[agg], labels)
        auc = auroc(scores_by_agg[agg], labels)
        verdict = ('strong (AUROC > 0.85)' if auc > 0.85
                   else 'usable (0.7-0.85, ensemble)' if auc > 0.70
                   else 'weak — rewrite rubric')
        print(f'{m:18s} {agg:>6s}  {rho:>+11.3f}  {auc:>7.3f}  {verdict}')

print()
print('Reading the table:')
print('  • The aggregation row with highest AUROC is the one to use as reward_fn.')
print('  • If ALL aggregations sit below AUROC 0.7, the rubric is not fit for purpose —')
print('    iterate on PRM_RUBRIC (try strict-arithmetic only, or logical-progression only)')
print('    and re-run this cell. Cache makes re-runs free per (rollout, ex) tuple.')
print('  • Spearman ρ is more forgiving than AUROC for monotone-but-noisy signals;')
print('    if ρ is high but AUROC is low you have signal at the extremes but not the middle.')


### 2.7 · Pass@k for the student — RS-SFT ceiling under the student's own distribution

Pass@k is the upper bound for any rejection-sampling SFT loop that uses *the student itself* as the rollout source. If `pass@K ≫ pass@1`, RS-SFT can extract that gap by training on the filtered correct rollouts. If `pass@K ≈ pass@1`, the student is saturated at the current temperature and distillation is the only path forward.

Uses the unbiased estimator from Chen et al. 2021 (HumanEval) and a bootstrap CI over examples.


In [ ]:
# Section 2.7 Pass@k — call-only. pass_at_k() is defined in Section 2 helpers.
# 64 problems × 8 rollouts at temp=0.7 ≈ 512 generations. Wall ~25-35 min.

for _required in ('pass_at_k', 'rollout'):
    if _required not in dir():
        raise NameError(f'{_required} is undefined. Run Section 2 helpers first.')

K       = 8
N_PASSK = 64
PASSK_T = 0.7

passk_correct = []  # (N_PASSK, K)
for ex in val.select(range(N_PASSK)):
    row = []
    for _ in range(K):
        r = rollout(student, build_prompt(ex['question']), temperature=PASSK_T)
        row.append(int(correct(extract_pred(r['text']), gold_answer(ex))))
    passk_correct.append(row)

passk_arr = np.asarray(passk_correct)

print(f'Student pass@k (temp={PASSK_T}, n={N_PASSK}, K={K}):')
rng = np.random.default_rng(0)
for k in [1, 4, K]:
    mean, per_ex = pass_at_k(passk_arr, k)
    boot = rng.choice(per_ex, size=(2000, len(per_ex)), replace=True).mean(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    print(f'  pass@{k:<2d} = {mean:.3f}  [boot95: {lo:.3f}, {hi:.3f}]')

print()
print('Reading pass@k:')
print('  • pass@1  ≈ student baseline at temp>0.')
print('  • pass@K  = RS-SFT ceiling using student-self rollouts.')
print('  • pass@K - pass@1 = headroom RS-SFT can extract via best-of-K filtering.')


### 2.7b · RS-SFT filter quality — does PRM approximate the verifier on STUDENT rollouts?

The §2.6b calibration scored `prm_judge` on **Cortex teacher** rollouts (large, well-formatted, mostly correct). Student rollouts at temp=0.7 are different — shorter, sometimes mid-sentence, lower base accuracy. The PRM may behave differently on them.

This cell runs the actual operational test: generate `RS_N × RS_K` student rollouts, compute both verifier ground-truth correctness and `prm_judge`, then sweep τ across `agg ∈ {min, mean}` and report:

- **kept / total** — fraction of rollouts that pass the PRM filter at this τ.
- **precision** — among kept rollouts, fraction that are actually verifier-correct. **The number that decides whether PRM-filtered SFT data is high quality.**
- **recall** — fraction of verifier-correct rollouts the PRM retains.
- **F1** — harmonic mean. The τ row with highest F1 is the best operating point.

### Decision rule for wiring `prm_judge` into RS-SFT

- **precision > 0.95 with recall > 0.5** → PRM filter is operationally usable. SFT on PRM-filtered rollouts even without gold answers.
- **precision tops out below 0.85** → PRM not selective enough; ensemble (Step E) or rewrite rubric (Step C).
- **`mean` usually beats `min` for F1** — strict-min throws away too many borderline-correct rollouts. Consistent with the §2.6b finding.

### Cost

`RS_N=32 × RS_K=4 = 128` rollouts → ~6 min student generation + ~10 min PRM scoring (cache amortises re-runs to free). Idempotent: re-running re-uses cached `rs_text` / `rs_correct` from the kernel.


In [ ]:
# Section 2.7b — RS-SFT filter quality on student rollouts.
# Generates a small student-rollout sample, computes verifier truth + PRM scores,
# sweeps τ across {min, mean}, reports precision/recall/F1 with Wilson CIs.

# Fail FAST on missing prereqs.
for _r in ('prm_judge', 'rollout', 'student', 'val', 'wilson_ci', 'build_prompt',
          'correct', 'extract_pred', 'gold_answer'):
    if _r not in dir():
        raise NameError(f'{_r} undefined. Run §2 helpers, §2.5 client, §2.6 PRM defs first.')

import time

RS_N = 32   # problems
RS_K = 4    # rollouts per problem
RS_T = 0.7  # sampling temperature

# Idempotent — keep student rollouts across re-runs (saves ~6 min on re-eval)
if 'rs_text' not in dir() or 'rs_correct' not in dir() or len(rs_text) != RS_N:
    rs_text, rs_correct = [], []
    print(f'Generating {RS_N} × {RS_K} = {RS_N*RS_K} student rollouts at temp={RS_T}...')
    t0 = time.time()
    for i, ex in enumerate(val.select(range(RS_N))):
        texts, corrs = [], []
        for _ in range(RS_K):
            r = rollout(student, build_prompt(ex['question']), temperature=RS_T)
            texts.append(r['text'])
            corrs.append(int(correct(extract_pred(r['text']), gold_answer(ex))))
        rs_text.append(texts)
        rs_correct.append(corrs)
        if (i+1) % 8 == 0: print(f'  {i+1}/{RS_N} problems done')
    print(f'Generation: {time.time()-t0:.1f}s wall')
else:
    print(f'Reusing cached rs_text, rs_correct ({sum(len(r) for r in rs_text)} rollouts)')

# Score with PRM (cached by rollout content + question + rubric + judge model)
print(f'\nScoring with prm_judge (cache size before: {len(_prm_cache)})...')
t0 = time.time()
rs_prm_mean, rs_prm_min = [], []
for i, ex in enumerate(val.select(range(RS_N))):
    pmean, pmin = [], []
    for text in rs_text[i]:
        pmean.append(prm_judge(text, ex, agg='mean'))
        pmin.append(prm_judge(text, ex, agg='min'))
    rs_prm_mean.append(pmean)
    rs_prm_min.append(pmin)
print(f'Scoring:    {time.time()-t0:.1f}s wall, cache size after: {len(_prm_cache)}')

# Flatten for easy filtering
flat_corr = [c for row in rs_correct  for c in row]
flat_mean = [p for row in rs_prm_mean for p in row]
flat_min  = [p for row in rs_prm_min  for p in row]
total = len(flat_corr)
n_correct_total = sum(flat_corr)
base_acc = n_correct_total / total

print()
print(f'Student baseline at temp={RS_T}: {n_correct_total}/{total} correct = {base_acc:.3f}')
print()
print('PRM filter quality at various τ:')
print(f'{"τ":>5} {"agg":>5}  {"kept/total":>11}  {"precision (P[correct|kept])":>32}  {"recall (P[kept|correct])":>30}  {"F1":>6}')
print('-' * 100)

best = {'mean': (-1, None), 'min': (-1, None)}
for agg_name, scores in [('mean', flat_mean), ('min', flat_min)]:
    for tau in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
        kept_mask = [1 if p >= tau else 0 for p in scores]
        n_kept = sum(kept_mask)
        if n_kept == 0:
            print(f'{tau:>5.2f} {agg_name:>5s}  {0:>3d}/{total:<3d}     (no rollouts pass threshold)')
            continue
        tp = sum(1 for k, c in zip(kept_mask, flat_corr) if k and c)
        fp = n_kept - tp
        prec = tp / n_kept
        rec  = tp / max(n_correct_total, 1)
        f1   = 2 * prec * rec / max(prec + rec, 1e-9)
        p_lo, p_hi = wilson_ci(tp, n_kept)
        r_lo, r_hi = wilson_ci(tp, n_correct_total)
        if f1 > best[agg_name][0]:
            best[agg_name] = (f1, (tau, prec, rec, f1, n_kept))
        print(f'{tau:>5.2f} {agg_name:>5s}  {n_kept:>3d}/{total:<3d}      '
              f'{prec:.3f} [{p_lo:.2f}, {p_hi:.2f}]   '
              f'{rec:.3f} [{r_lo:.2f}, {r_hi:.2f}]   '
              f'{f1:.3f}')

print()
print('Best operating points:')
for agg_name, (f1, info) in best.items():
    if info is None: continue
    tau, prec, rec, f1, n_kept = info
    print(f'  {agg_name}: τ={tau:.1f} → precision={prec:.3f}, recall={rec:.3f}, F1={f1:.3f}, '
          f'kept={n_kept}/{total}')

print()
print('Reading the table:')
print('  • precision = "if I SFT on PRM-kept rollouts, how clean is the training data?"')
print('  • recall    = "what fraction of correct rollouts does the filter keep?"')
print('  • Compare against the trivial verifier filter, which has precision=1.0 by definition,')
print('    recall=1.0, F1=1.0 — but requires gold answers (defeats the point of RS-SFT in production).')
print()
print('Decision:')
print(f'  • If best precision > 0.95 AND recall > 0.5 → wire prm_judge into RS-SFT (Step F1)')
print(f'    and unified_step reward_fn (Step F2). Calibration is operationally sufficient.')
print(f'  • If precision tops out below 0.85 → ensemble (Step E) or rubric iteration (Step C)')
print(f'    before wiring in. Cache makes rubric iteration free per (rollout, ex) tuple.')
print(f'  • If precision is high but recall is low (< 0.3) → larger rollout pool (raise RS_K)')
print(f'    to compensate. Filter quality is fine, just need more candidates.')


### Local-teacher accuracy — the OPD ceiling

Cortex models are external references. The **actually relevant** ceiling for OPD on this notebook is the local `external_teacher` (Mixtral-8x7B-Instruct-v0.1, 4-bit) — that's the model whose per-token logprobs are training the student. Run the same `evaluate(...)` against it so we have:

- Student baseline (`acc0`, local Mistral-7B-v0.2)
- **OPD teacher (local Mixtral-8x7B)** — the SFT-RS ceiling reachable with this teacher
- Cortex mistral-large2 / llama4-maverick — external references for comparison

The OPSD-trained student in Section 7 should sit between the baseline and the local Mixtral ceiling.


In [ ]:
# Evaluate the local OPD teacher (Mixtral-8x7B) on the same N_VAL val subset.
# Same prompt, same metric, same n — apples-to-apples and PAIRED with student_correct.

# Fail FAST before the 3-5 min Mixtral generation loop.
for _required in ('fmt_acc', 'mcnemar_p', 'bootstrap_gap_ci'):
    if _required not in dir():
        raise NameError(
            f'{_required} is undefined. Run the eval-stats-utilities cell first.')
if 'student_correct' not in dir():
    raise NameError('student_correct is undefined. Run the Section 3 baseline cell first.')

if 'external_teacher' not in globals():
    raise RuntimeError('external_teacher not loaded — run Section 4 OPD cell first.')

acc_mixtral_local, mixtral_correct = evaluate(external_teacher, val, n=N_VAL)
print(f'OPD teacher (local Mixtral-8x7B 4-bit)  {fmt_acc(mixtral_correct)}')

print()
print('=== full Section 1 ceiling reference ===')
print(f'  student baseline (Mistral-7B-v0.2)        {fmt_acc(student_correct)}')
print(f'  OPD teacher (local Mixtral-8x7B)          {fmt_acc(mixtral_correct)}')
for m, arr in cortex_correct.items():
    print(f'  Cortex {m:22s}        {fmt_acc(arr)}')
print()

# Paired: local Mixtral vs student
mn = mcnemar_p(student_correct, mixtral_correct)
gap, glo, ghi = bootstrap_gap_ci(student_correct, mixtral_correct)
print(f'Mixtral-8x7B vs student: gap={gap:+.3f} [boot95: {glo:+.3f}, {ghi:+.3f}]   '
      f'McNemar χ²={mn["chi2"]:.2f} p={mn["p"]:.4f}   '
      f'(student-only-right={mn["b10"]}, teacher-only-right={mn["b01"]})')
print()
print('OPSD-clip post-Section 7 student should land between acc0 and the local Mixtral ceiling.')
print('Re-run evaluate(student, val, n=N_VAL) after Section 7 and compare against student_correct')
print('with mcnemar_p(student_correct, post_correct) to test for paired improvement.')


## 4 · Section 3 — OPD (real same-family teacher: Mixtral-8x7B)

`mistralai/Mixtral-8x7B-Instruct-v0.1` shares the 32 K Mistral tokenizer with `Mistral-7B-Instruct-v0.2`, so the per-token reverse KL is well-defined position-by-position. (Mistral-7B-v0.3 has an extended 32 768 vocab and would *not* match — that's why we use v0.2 here.) Loaded **lazily and in 4-bit** (~24 GB), kept frozen, used only for forward passes.


In [ ]:
# Lazy-load Mixtral-8x7B-Instruct as the external same-family teacher.
# 4-bit nf4: ~24 GB. With Mistral-7B LoRA student (~5 GB) you need ~30 GB headroom + activations.
# A100-80GB / H100-80GB GPU pools fit comfortably.
EXTERNAL_TEACHER_NAME = 'mistralai/Mixtral-8x7B-Instruct-v0.1'

if 'external_teacher' not in globals():
    print(f'Loading {EXTERNAL_TEACHER_NAME} (4-bit, frozen)...')
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
                              bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    external_teacher = AutoModelForCausalLM.from_pretrained(
        EXTERNAL_TEACHER_NAME, quantization_config=bnb, device_map='auto')
    external_teacher.eval()
    if hasattr(external_teacher, 'gradient_checkpointing_disable'):
        external_teacher.gradient_checkpointing_disable()
    for p in external_teacher.parameters(): p.requires_grad_(False)

    # Enforce same-family precondition at runtime (token-aligned KL requires identical vocab).
    tok_T = AutoTokenizer.from_pretrained(EXTERNAL_TEACHER_NAME)
    if tok.get_vocab() != tok_T.get_vocab():
        raise RuntimeError(
            'Tokenizer mismatch between student (Mistral-7B-Instruct-v0.2) and teacher '
            '(Mixtral-8x7B-Instruct-v0.1). Switch student to Mistral-7B-Instruct-v0.2 '
            'for byte-identical 32K vocab.')
    print('vocab matched ✓ — OPD same-family precondition holds')
    print(f'teacher params: {sum(p.numel() for p in external_teacher.parameters())/1e9:.1f} B')

def opd_step(student, teacher, prompt: str, kl_clip=None) -> dict:
    """Section 3 OPD: student samples on-policy, teacher provides per-token reverse KL."""
    r = rollout(student, prompt, temperature=0.7)
    logp_S = logprobs_at(student, r['full'], r['plen'])           # grad enabled
    with torch.no_grad():
        logp_T = logprobs_at(teacher, r['full'], r['plen'])       # frozen
    kl, kl_raw = per_token_reverse_kl(logp_T, logp_S, clip=kl_clip)
    loss = kl.mean()
    loss.backward()
    return {'loss': loss.item(), 'kl_raw': kl_raw.cpu().float().numpy(),
            'gen_text': r['text']}

# Single demonstrative OPD step — record per-token KL distribution
student.zero_grad(set_to_none=True)
demo = opd_step(student, external_teacher, build_prompt(ds[0]['question']))
print(f"OPD: loss = {demo['loss']:.4f}")
print(f"     per-token KL — mean {demo['kl_raw'].mean():.4f}, max {demo['kl_raw'].max():.4f}")
print(f"     max/mean = {demo['kl_raw'].max()/max(demo['kl_raw'].mean(),1e-8):.1f}  (lower → diffuse)")


## 5 · Section 4 — OPSD (on-policy self-distillation with privileged info)

**Teacher is the student itself**, but the teacher gets the **ground-truth answer** in its context. Student samples without the GT in context. The teacher's distribution sharpens around the answer-consistent tokens, especially at "pivot tokens" — exactly the concentration mechanism Section 5 of the post calls out.

In [ ]:
# self_teacher was already created above as a separate frozen copy of the base model.
# Defensive: ensure no gradient checkpointing on the frozen teacher — it's only used for
# forward passes under torch.no_grad(), where checkpointing is wasted compute and triggers
# 'None of the inputs have requires_grad=True' warnings.
if hasattr(self_teacher, 'gradient_checkpointing_disable'):
    self_teacher.gradient_checkpointing_disable()
self_teacher.eval()
for p in self_teacher.parameters(): p.requires_grad_(False)

def opsd_step(student, self_teacher, ex, lr=5e-6, kl_clip: Optional[float] = None) -> dict:
    bare_prompt = build_prompt(ex['question'])
    gt_prompt   = build_prompt(ex['question'], answer=gold_answer(ex))
    r = rollout(student, bare_prompt, temperature=0.7)
    bare_ids = tok(bare_prompt, return_tensors='pt').input_ids[0].to(DEVICE)
    gt_ids   = tok(gt_prompt,   return_tensors='pt').input_ids[0].to(DEVICE)
    # Teacher sees: gt_prompt + student-generated tokens
    logp_S = logprobs_at(student, r['full'], r['plen'])
    with torch.no_grad():
        logp_T = logprobs_at(self_teacher, r['full'], r['plen'],
                             teacher_prompt_ids=gt_ids)
    kl, kl_raw = per_token_reverse_kl(logp_T, logp_S, clip=kl_clip)
    loss = kl.mean()
    loss.backward()
    return {'loss': loss.item(), 'kl_raw': kl_raw.cpu().float().numpy(),
            'gen_text': r['text']}

student.zero_grad(set_to_none=True)
demo_opsd = opsd_step(student, self_teacher, ds[0])
print(f"OPSD: loss = {demo_opsd['loss']:.4f}")
print(f"      per-token KL — mean {demo_opsd['kl_raw'].mean():.4f}, max {demo_opsd['kl_raw'].max():.4f}")

## 6 · Section 5 — Gradient geometry: *diffuse* OPD vs *concentrated* OPSD

Run a handful of un-optimized steps for each method, accumulate per-token KL contributions, and plot the histogram. The post's prediction: OPSD has a **fat right tail** (a few pivot tokens dominate), OPD does not.

In [ ]:
@torch.no_grad()
def collect_kl(method: str, n: int = 6):
    arr = []
    for i in range(n):
        ex = ds[i]
        if method == 'OPD':
            r = rollout(student, build_prompt(ex['question']), temperature=0.7)
            logp_S = logprobs_at(student, r['full'], r['plen'])
            logp_T = logprobs_at(external_teacher, r['full'], r['plen'])
        else:  # OPSD
            bare = build_prompt(ex['question'])
            gt   = build_prompt(ex['question'], answer=gold_answer(ex))
            r = rollout(student, bare, temperature=0.7)
            gt_ids = tok(gt, return_tensors='pt').input_ids[0].to(DEVICE)
            logp_S = logprobs_at(student, r['full'], r['plen'])
            logp_T = logprobs_at(self_teacher, r['full'], r['plen'],
                                 teacher_prompt_ids=gt_ids)
        kl, _ = per_token_reverse_kl(logp_T, logp_S)
        arr.append(kl.cpu().float().numpy())
    return np.concatenate(arr)

kl_opd  = collect_kl('OPD',  n=6)
kl_opsd = collect_kl('OPSD', n=6)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, kl, name in zip(axes, [kl_opd, kl_opsd],
                        ['OPD (Mixtral-8x7B teacher)', 'OPSD (self + GT context)']):
    ax.hist(kl, bins=40, alpha=0.85)
    ax.set_title(f'{name}\nmean={kl.mean():.3f}  max={kl.max():.3f}  '
                 f'max/mean={kl.max()/max(kl.mean(),1e-8):.1f}')
    ax.set_xlabel('per-token reverse KL'); ax.set_ylabel('count'); ax.set_yscale('log')
plt.tight_layout(); plt.show()
print('Section 5 prediction:')
print(f'  OPD  max/mean = {kl_opd.max()/max(kl_opd.mean(),1e-8):.1f}   (diffuse — calibrated teacher)')
print(f'  OPSD max/mean = {kl_opsd.max()/max(kl_opsd.mean(),1e-8):.1f}  (concentrated — pivot tokens)')


## 7 · Section 5 — KL clipping demonstration

Train OPSD twice for ~50 steps each: **without** point-wise KL clip → expect collapse (val acc drops, generations degenerate). **With** clip → stable.

(50 steps is short but enough to see the divergence; the OPSD paper reports collapse within ~100.)

In [ ]:
# OPSD trainer — snapshots LoRA adapter weights only (not the 4-bit base) so the
# no-clip and clip runs start from the same point. deepcopy(student) won't work on
# 4-bit + PEFT models, so we save/restore LoRA state_dict instead.
from torch.optim import AdamW
from peft.utils.save_and_load import get_peft_model_state_dict, set_peft_model_state_dict

# Snapshot the *current* LoRA adapter weights (run this once, before the first train).
lora_init_state = {k: v.detach().clone() for k, v in get_peft_model_state_dict(student).items()}
print(f'snapshotted {sum(v.numel() for v in lora_init_state.values())/1e6:.1f} M LoRA params')

def restore_lora_init():
    set_peft_model_state_dict(student, {k: v.clone() for k, v in lora_init_state.items()})

def train_opsd(steps: int, kl_clip, lr: float = 5e-6, eval_every: int = 25):
    restore_lora_init()                            # same starting point each run
    student.train()
    opt = AdamW([p for p in student.parameters() if p.requires_grad], lr=lr)
    losses, accs = [], []
    for step in range(steps):
        ex = ds[step % len(ds)]
        opt.zero_grad(set_to_none=True)
        out = opsd_step(student, self_teacher, ex, kl_clip=kl_clip)
        gn = torch.nn.utils.clip_grad_norm_(
                [p for p in student.parameters() if p.requires_grad],
                max_norm=1e9).item()
        opt.step()
        losses.append((out['loss'], gn))
        if (step + 1) % eval_every == 0 or step == steps - 1:
            accs.append((step + 1, evaluate(student, val, n=12)))
            print(f'  step {step+1:3d}  loss={out["loss"]:.3f}  ‖g‖={gn:.2f}  acc(12)={accs[-1][1]:.3f}')
    student.eval()
    return losses, accs

print('--- OPSD without KL clipping (expect collapse) ---')
loss_no_clip, acc_no_clip = train_opsd(steps=50, kl_clip=None, lr=5e-6, eval_every=25)
print('\n--- OPSD with KL clipping (kl_clip = 1.0) ---')
loss_clip,    acc_clip    = train_opsd(steps=50, kl_clip=1.0,  lr=5e-6, eval_every=25)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot([l for l,_ in loss_no_clip], label='no clip',  alpha=0.85)
ax[0].plot([l for l,_ in loss_clip],    label='clip=1.0', alpha=0.85)
ax[0].set_title('OPSD loss'); ax[0].set_xlabel('step'); ax[0].legend()
ax[1].plot([g for _,g in loss_no_clip], label='no clip',  alpha=0.85)
ax[1].plot([g for _,g in loss_clip],    label='clip=1.0', alpha=0.85)
ax[1].set_title('OPSD gradient norm  ‖g‖'); ax[1].set_xlabel('step'); ax[1].set_yscale('log'); ax[1].legend()
plt.tight_layout(); plt.show()
print('Final val acc (12 problems): no-clip', acc_no_clip[-1][1], '| clip', acc_clip[-1][1])

## 8 · Section 6 — Gradient taxonomy

| Method | Sparsity | Bias | Concentration | Defense needed |
|---|---|---|---|---|
| **RL** (GRPO-style) | Sparse, broadcast | Unbiased (zero-mean within group) | n/a | none — destructive interference |
| **SFT (teacher)** | Dense | Biased toward data | Diffuse (many tokens, small bias each) | none — bias is unconcentrated |
| **OPD** (same-family) | Dense | Biased toward teacher | Diffuse (calibrated teacher) | none |
| **OPSD** (self + GT) | Dense | Biased toward GT-conditioned self | **Concentrated** (pivot tokens dominate) | KL clipping required |

The histograms in Section 5 above are the empirical evidence for the *concentration* row.

## 9 · Section 7 — Unified meta-algorithm

All four methods are special cases of a single token-level policy gradient parameterized by:

- `α ∈ [0,1]` — how on-policy the sampling distribution is. `α=0` → sample from teacher (SFT), `α=1` → sample from student (RL / OPD / OPSD).
- `λ ∈ [0,1]` — share of the per-token advantage from a teacher KL versus an outcome reward. `λ=0` → pure RL, `λ=1` → pure distillation.
- `π_T(c_T)` — the teacher policy and *what context it sees*: external same-family teacher (OPD), self with no extra info (no-op), self with demo (SDFT), self with GT answer (OPSD).

The function below implements that single unified loss and shows each method as a setting of the knobs.

In [ ]:
def unified_step(student, ex,
                 alpha: float, lam: float,
                 teacher_policy: Callable, reward_fn: Callable, sampler,
                 kl_clip: Optional[float] = None) -> dict:
    prompt = build_prompt(ex['question'])
    r = rollout(sampler, prompt, temperature=0.7)
    logp_S = logprobs_at(student, r['full'], r['plen'])
    with torch.no_grad():
        logp_T = teacher_policy(r, ex)
    kl_per_tok, _ = per_token_reverse_kl(logp_T, logp_S, clip=kl_clip)
    kl_loss = kl_per_tok.mean()
    R = reward_fn(r['text'], ex)
    gen_logp = logp_S.gather(-1, r['full'][r['plen']:].unsqueeze(-1)).squeeze(-1)
    rl_loss  = -(R * gen_logp).mean()
    loss = lam * kl_loss + (1 - lam) * rl_loss
    return {'loss': loss, 'kl': kl_loss.item(), 'rl': rl_loss.item(),
            'reward': R, 'alpha_used': alpha}

def reward_fn(text, ex):
    return 1.0 if correct(extract_pred(text), gold_answer(ex)) else 0.0

def teacher_external(r, ex):
    return logprobs_at(external_teacher, r['full'], r['plen'])

def teacher_self_gt(r, ex):
    gt_ids = tok(build_prompt(ex['question'], answer=gold_answer(ex)),
                 return_tensors='pt').input_ids[0].to(DEVICE)
    return logprobs_at(self_teacher, r['full'], r['plen'], teacher_prompt_ids=gt_ids)

configs = {
    'SFT (teacher samples)': dict(alpha=0.0, lam=1.0, teacher_policy=teacher_external,
                                    sampler=external_teacher, kl_clip=None),
    'RL (outcome only)':      dict(alpha=1.0, lam=0.0, teacher_policy=teacher_external,
                                    sampler=student,           kl_clip=None),
    'OPD (Mixtral teacher)':  dict(alpha=1.0, lam=1.0, teacher_policy=teacher_external,
                                    sampler=student,           kl_clip=None),
    'OPSD (self+GT teacher)': dict(alpha=1.0, lam=1.0, teacher_policy=teacher_self_gt,
                                    sampler=student,           kl_clip=1.0),
}

ex = ds[0]
for name, cfg in configs.items():
    student.zero_grad(set_to_none=True)
    out = unified_step(student, ex, reward_fn=reward_fn, **cfg)
    print(f'{name:28s} α={cfg["alpha"]:.1f} λ={cfg["lam"]:.1f}  '
          f'KL={out["kl"]:.3f} RL={out["rl"]:.3f} R={out["reward"]:.0f}')


## 10 · OPD vs OPSD — trend summary

Both methods optimise the same per-token reverse-KL objective `KL(π_S || π_T)` on student-sampled rollouts. The only knob that changes between them is **who plays the teacher** — and that single choice flips the gradient geometry, the failure modes, and the right hyperparameter regime.

### Side-by-side

| Axis | OPD (external same-family teacher) | OPSD (self + privileged GT in context) |
|---|---|---|
| Teacher identity | Different, larger, frozen (Mixtral-8x7B here) | Same model as student, frozen, with GT answer in its prompt |
| Precondition | Identical tokenizer / vocab — enforced in Section 4 | Trivially holds — single model, two copies |
| Information advantage | Capacity / pretraining gap | Privileged context (GT visible to teacher only) |
| Per-token KL shape | **Diffuse** — signal spread across many tokens | **Fat-tailed / spiky** — mass concentrated at a few "pivot" tokens |
| `max(KL) / mean(KL)` | Low (close to 1) | High — the concentration ratio Section 5 predicts |
| Gradient norm distribution | Smooth, bounded | A handful of dominant tokens, others ≈ 0 |
| Failure mode without clipping | Slow, but stable | **Collapses** — pivot-token KL explodes, LoRA gradient blows up |
| Required mitigation | Standard LR scheduling | **Per-token KL clipping** (`kl_clip=1.0` here) |
| Compute cost per step | 2 model forwards, one of them on a much bigger model | 2 forwards on the same model, one extra teacher prefix (GT) |
| Data dependency | None beyond the prompt | Requires gold answers (otherwise OPSD is just self-distillation) |
| Best fit | Distilling a strong base into a small student | Concentrating an existing model on the right tokens, no external teacher available |

### What we actually observed in this notebook

- **Section 4 OPD step.** Single Mixtral-teacher OPD step ran cleanly. Per-token KL distribution was diffuse: `max/mean` ratio low, no token dominating the gradient. Loss values were well-conditioned without any clipping.
- **Section 5 OPSD step.** Same student, but the frozen self-teacher saw the GT answer. Per-token KL was visibly fat-tailed — a small number of tokens carried most of the mass. This is the empirical signature of the concentration mechanism: the teacher "knows" exactly where the answer-determining tokens are because the GT is in its context.
- **Section 6 distribution comparison.** Plotted side-by-side, OPD KL is roughly unimodal and tight; OPSD KL has a long right tail with the `max/mean` ratio noticeably higher — quantifying Section 5's claim instead of just asserting it.
- **Section 7 KL-clipping ablation.** With no clip, OPSD's pivot tokens drive the gradient norm into a regime that destabilises LoRA updates. Setting `kl_clip=1.0` truncates only the tail without touching the bulk, and OPSD becomes well-behaved. OPD with the same clip is essentially unchanged because there was nothing in the tail to clip — confirming the failure mode is OPSD-specific.
- **Section 9 unified meta-algorithm.** SFT, RL, OPD, OPSD all collapse to four `(α, λ, π_T)` settings of one `unified_step`. OPD and OPSD only differ in the `teacher_policy` argument; the dispatcher does the rest. Re-running Section 9 with `kl_clip=1.0` *only* on the OPSD row reproduces the Section 7 result inline.

### One-sentence trend

**OPD spreads a smaller, smoother gradient across tokens; OPSD concentrates a larger, spikier gradient on the pivot tokens where the GT actually disambiguates the rollout — so OPSD wins on signal-to-noise per token but only if you clip the tail.**


## 11 · Takeaways

- **Same-family OPD** runs end-to-end: Mistral-7B student (LoRA, trainable) + Mixtral-8x7B teacher (4-bit, frozen). Tokenizer-match assertion enforces the precondition.
- **OPSD** also runs: same Mistral-7B frozen with the GT answer in its system prompt acts as the privileged-info teacher.
- **OPD vs OPSD KL distributions** in Section 6 are the empirical version of Section 5: OPD diffuse, OPSD fat-tailed. The `max/mean` ratio quantifies the concentration claim.
- **KL clipping** in Section 7 stays the bright line — without it OPSD collapses; with `kl_clip=1.0` it stays stable. Same code, just a flag.
- **Unified meta-algorithm** in Section 9 reproduces SFT, RL, OPD, OPSD as four settings of `(α, λ, π_T)` — the OPD slot is now real, not pseudocode.
- **Memory budget**: Mixtral 4-bit ~24 GB + Mistral-7B LoRA ~5 GB + activations + KV cache ≈ 35–45 GB peak. Fits comfortably on an A100-80GB / H100 GPU pool.
